In [2]:
"""
test_telegram.py
==================
สคริปต์ทดสอบส่งข้อความเข้า Telegram โดยพิมพ์ข้อความที่ต้องการผ่าน Terminal
"""
import os
import requests
from dotenv import load_dotenv

load_dotenv()

TELEGRAM_BOT_TOKEN = os.getenv("TELEGRAM_BOT_TOKEN")
TELEGRAM_CHAT_ID = os.getenv("TELEGRAM_CHAT_ID")

def is_configured():
    return bool(TELEGRAM_BOT_TOKEN and TELEGRAM_CHAT_ID)

def send(message):
    url = f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendMessage"
    payload = {
        "chat_id": TELEGRAM_CHAT_ID,
        "text": message,
        "parse_mode": "Markdown"
    }
    
    try:
        response = requests.post(url, json=payload)
        response.raise_for_status()
        return True
    except Exception as e:
        print(f"❌ เกิดข้อผิดพลาด: {e}")
        return False

# ==========================================
# ส่วนรันโปรแกรม
# ==========================================
if __name__ == "__main__":
    print(f"Configured: {is_configured()}")

    if not is_configured():
        print("❌ ยังไม่ได้ตั้งค่า TELEGRAM_BOT_TOKEN หรือ TELEGRAM_CHAT_ID ในไฟล์ .env")
    else:
        # ให้โปรแกรมรอรับข้อความจากคีย์บอร์ด
        custom_message = input("💬 พิมพ์ข้อความที่ต้องการส่งไป Telegram: ")
        
        print("กำลังส่งข้อความ...")
        ok = send(custom_message) # ส่งข้อความที่คุณพิมพ์เข้าไป
        print(f"สถานะการส่ง: {'สำเร็จ ✅' if ok else 'ล้มเหลว ❌'}")



Configured: True
กำลังส่งข้อความ...
สถานะการส่ง: สำเร็จ ✅


In [4]:
import os
from datetime import datetime
import requests
from dotenv import load_dotenv

# โหลดค่าจากไฟล์ .env
load_dotenv()

# ดึงค่าพร้อมตัดช่องว่างส่วนเกินออกอัตโนมัติ
TELEGRAM_BOT_TOKEN = (
    os.getenv("TELEGRAM_BOT_TOKEN", "").replace(" ", "").strip()
)
TELEGRAM_CHAT_ID = os.getenv("TELEGRAM_CHAT_ID", "").replace(" ", "").strip()


def is_configured():
  return bool(TELEGRAM_BOT_TOKEN and TELEGRAM_CHAT_ID)


def send_telegram(message):
    """ส่งข้อความเข้า Telegram"""
    # ล้างอักขระแปลกปลอมที่อาจติดมาจาก copy-paste
    clean_token = TELEGRAM_BOT_TOKEN.strip().replace("[", "").replace("]", "")
    clean_chat_id = TELEGRAM_CHAT_ID.strip().replace("[", "").replace("]", "")

    url = f"https://api.telegram.org/bot{clean_token}/sendMessage"
    
    payload = {
        "chat_id": clean_chat_id,
        "text": message,
        "parse_mode": "Markdown"
    }

    try:
        response = requests.post(url, json=payload)
        response.raise_for_status()
        return True
    except Exception as e:
        print(f"❌ เกิดข้อผิดพลาดในการส่ง: {e}")
        if hasattr(e, 'response') and e.response is not None:
            print(f"🔍 รายละเอียดจาก Telegram: {e.response.text}")
        return False


# ============================================================
# MOCK DATA & BRIDGES (ข้อมูลจำลองสำหรับทดสอบ)
# ============================================================
class MockCreds:

  sandbox = True


class MockBridge:

  def get_account_summary(self):
    return {
        "total_asset_currency": "USD",
        "total_net_liquidation_value": 125432.50,
        "total_cash_balance": 45200.10,
        "total_market_value": 80232.40,
        "total_unrealized_profit_loss": 12450.75,
        "total_day_profit_loss": -320.50,
        "maintenance_margin": 15000.00,
        "open_margin_calls": [],
    }

  def get_positions(self):
    return [
        {
            "symbol": "AAPL",
            "quantity": 150,
            "last_price": 185.25,
            "market_value": 27787.50,
            "unrealized_profit_loss": 3250.00,
        },
        {
            "symbol": "NVDA",
            "quantity": 50,
            "last_price": 880.00,
            "market_value": 44000.00,
            "unrealized_profit_loss": 10200.50,
        },
        {
            "symbol": "TSLA",
            "quantity": 40,
            "last_price": 211.12,
            "market_value": 8444.80,
            "unrealized_profit_loss": -999.75,
        },
    ]

  def list_open_orders(self):
    return ["order_1", "order_2"]


creds = MockCreds()
bridge = MockBridge()


# ============================================================
# FORMAT FUNCTIONS
# ============================================================
def money(x):
  try:
    return f"${float(x):,.2f}"
  except Exception:
    return "-"


def signed_money(x):
  try:
    value = float(x)
    return f"+${value:,.2f}" if value >= 0 else f"-${abs(value):,.2f}"
  except Exception:
    return "-"


# ============================================================
# BUILD TELEGRAM REPORT MESSAGE
# ============================================================
def build_dashboard_message():
  """ประกอบข้อความแดชบอร์ดให้อยู่ในรูปแบบที่แสดงผลสวยงามบน Telegram"""
  balance = bridge.get_account_summary()
  positions = bridge.get_positions()
  updated = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
  currency = balance["total_asset_currency"]
  environment = "SANDBOX / PAPER" if creds.sandbox else "LIVE"

  lines = []
  lines.append("📊 *WEBULL PORTFOLIO DASHBOARD*")
  lines.append("```")  # เปิด Code block ให้ตัวอักษรเรียงตรงกันบนมือถือ
  lines.append(f"⏱️ Updated     : {updated}")
  lines.append(f"🌎 Environment : {environment}")
  lines.append(f"💵 Currency    : {currency}")
  lines.append("-" * 36)
  lines.append("ACCOUNT SUMMARY")
  lines.append("-" * 36)
  lines.append(
      f"💰 Net Liq  : {money(balance['total_net_liquidation_value'])}"
  )
  lines.append(f"💵 Cash     : {money(balance['total_cash_balance'])}")
  lines.append(f"📊 Market   : {money(balance['total_market_value'])}")
  lines.append(
      f"📈 P/L      :"
      f" {signed_money(balance['total_unrealized_profit_loss'])}"
  )
  lines.append(
      f"📅 Day P/L  : {signed_money(balance['total_day_profit_loss'])}"
  )
  lines.append(f"🛡️ Margin   : {money(balance['maintenance_margin'])}")
  lines.append("-" * 36)
  lines.append("PORTFOLIO")
  lines.append("-" * 36)
  lines.append(f"{'Symbol':<6} {'Qty':>5} {'Price':>8} {'P/L':>12}")
  lines.append("-" * 36)

  for p in positions:
    symbol = p["symbol"]
    qty = int(float(p["quantity"]))
    price = money(p["last_price"])
    pl = signed_money(p["unrealized_profit_loss"])
    lines.append(f"{symbol:<6} {qty:>5} {price:>8} {pl:>12}")

  lines.append("-" * 36)
  lines.append("STATUS")
  lines.append("-" * 36)
  lines.append(f"🟢 Open Orders : {len(bridge.list_open_orders())}")
  margin_call = "YES" if balance["open_margin_calls"] else "None"
  lines.append(f"🟢 Margin Call : {margin_call}")
  lines.append("🟢 Account     : Connected")
  lines.append("```")  # ปิด Code block

  return "\n".join(lines)


# ============================================================
# MAIN RUN
# ============================================================
if __name__ == "__main__":
  if not is_configured():
    print(
        "❌ กรุณาตั้งค่า TELEGRAM_BOT_TOKEN และ TELEGRAM_CHAT_ID ในไฟล์ .env"
        " ก่อน"
    )
  else:
    print("กำลังสร้างข้อความแดชบอร์ด...")
    dashboard_msg = build_dashboard_message()

    print("\nกำลังส่งข้อความเข้ากลุ่ม Telegram...")
    if send_telegram(dashboard_msg):
      print("ส่งสำเร็จเรียบร้อย! ✅ (เปิดเช็กข้อความในกลุ่ม Telegram ได้เลย)")
    else:
      print("ส่งไม่สำเร็จ ❌")

กำลังสร้างข้อความแดชบอร์ด...

กำลังส่งข้อความเข้ากลุ่ม Telegram...
ส่งสำเร็จเรียบร้อย! ✅ (เปิดเช็กข้อความในกลุ่ม Telegram ได้เลย)
